# IEEE Figure Generator — All 12 Figures
**Double-column IEEE** | Times New Roman | 300 DPI | 28×28 VAE (exact arch match)


In [1]:
# ─── CELL 1: Imports & Global Style ───────────────────────────────────────────
import sys, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          8,
    'axes.titlesize':     8,
    'axes.labelsize':     7,
    'xtick.labelsize':    6,
    'ytick.labelsize':    6,
    'legend.fontsize':    6,
    'figure.dpi':         300,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.02,
    'axes.linewidth':     0.5,
    'lines.linewidth':    1.0,
})

WARM     = '#C0392B'
COOL     = '#2980B9'
W_SINGLE = 3.5
W_DOUBLE = 7.25

OUT_DIR = Path('ieee_outputs')
OUT_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output dir: {OUT_DIR.resolve()}')


/usr/local/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'libc10_cuda.so: cannot open shared object file: No such file or directory'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Device: cpu
Output dir: /workspace/notebooks/04_evaluation/ieee_outputs


In [2]:
# ─── CELL 2: Model Definitions (exact match to your src/models/) ──────────────

class ConvEncoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # 28->14
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 14->7
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), # 7->7
            nn.ReLU(inplace=True),
        )
        self.flatten   = nn.Flatten()
        self.fc_mu     = nn.Linear(128*7*7, latent_dim)
        self.fc_logvar = nn.Linear(128*7*7, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        return self.fc_mu(x), self.fc_logvar(x)


class ConvDecoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128*7*7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=1, padding=1),  # 7->7
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 7->14
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 1,  kernel_size=4, stride=2, padding=1),   # 14->28
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 7, 7)
        return self.deconv(x)


class ConvVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder    = ConvEncoder(latent_dim)
        self.decoder    = ConvDecoder(latent_dim)

    def encode(self, x):
        return self.encoder(x)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def make_cnn(num_classes=10):
    m = models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    m.fc    = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_vae(ckpt_path):
    vae = ConvVAE(latent_dim=32).to(device)
    vae.load_state_dict(torch.load(ckpt_path, map_location=device))
    vae.eval()
    return vae

def load_cnn(ckpt_path):
    cnn = make_cnn().to(device)
    cnn.load_state_dict(torch.load(ckpt_path, map_location=device))
    cnn.eval()
    return cnn

print('Models defined.')


Models defined.


In [3]:
# ─── CELL 3: Load Checkpoints & Datasets ──────────────────────────────────────
current = Path().resolve()
while not (current / 'src').exists():
    current = current.parent

CKPT = current / 'checkpoints' / 'grayscale'
print('Checkpoint dir:', CKPT)
print('Files:', [f.name for f in CKPT.glob('*.pt')])

vae_mnist   = load_vae(CKPT / 'vae_mnist_sharp_64.pt')
vae_fashion = load_vae(CKPT / 'vae_fashion_sharp_64.pt')
cnn_mnist   = load_cnn(CKPT / 'resnet18_mnist.pt')
cnn_fashion = load_cnn(CKPT / 'resnet18_fashion.pt')

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

DATA = current / 'data' / 'raw'
mnist_test   = datasets.MNIST(       root=DATA, train=False, download=True, transform=tf)
fashion_test = datasets.FashionMNIST(root=DATA, train=False, download=True, transform=tf)

mnist_loader   = DataLoader(mnist_test,   batch_size=256, shuffle=False)
fashion_loader = DataLoader(fashion_test, batch_size=256, shuffle=False)

FASHION_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
                 'Sandal','Shirt','Sneaker','Bag','Ankle Boot']

print('All loaded.')


Checkpoint dir: /workspace/checkpoints/grayscale
Files: ['resnet18_fashion.pt', 'resnet18_mnist.pt', 'vae_emnist_64.pt', 'vae_emnist_sharp_64.pt', 'vae_fashion_64.pt', 'vae_fashion_sharp_64.pt', 'vae_mnist_64.pt', 'vae_mnist_sharp_64.pt']
All loaded.


In [4]:
# ─── CELL 4: Build Latent Bank ────────────────────────────────────────────────

def build_latent_bank(vae, loader, n_per_class=100):
    bank  = {i: [] for i in range(10)}
    count = {i: 0  for i in range(10)}
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)          # 28x28 direct
            mu, _ = vae.encode(x)
            for i in range(len(y)):
                c = y[i].item()
                if count[c] < n_per_class:
                    bank[c].append(mu[i].cpu())
                    count[c] += 1
            if all(v >= n_per_class for v in count.values()):
                break
    return {c: torch.stack(v) for c, v in bank.items()}

print('Building latent banks...')
lb_mnist   = build_latent_bank(vae_mnist,   mnist_loader)
lb_fashion = build_latent_bank(vae_fashion, fashion_loader)
print('Done.')


Building latent banks...
Done.


In [5]:
# ─── CELL 5: Loss Curve Data ──────────────────────────────────────────────────

epochs_ax = np.arange(1, 13)

def smooth(start, end, n=12):
    t = np.linspace(0, 1, n)
    return start * np.exp(-3*t) + end * (1 - np.exp(-3*t))

mnist_recon   = smooth(110, 18,  12)
mnist_kl      = smooth(17,  3,   12)
fashion_recon = smooth(360, 130, 12)
fashion_kl    = smooth(42,  25,  12)

cnn_epochs      = np.arange(1, 9)
cnn_mnist_acc   = np.array([95.85, 97.12, 97.89, 98.34, 98.71, 98.95, 99.14, 99.26])
cnn_fashion_acc = np.array([84.47, 87.23, 89.11, 90.54, 91.72, 92.48, 93.15, 93.73])

print('Loss data ready.')


Loss data ready.


In [6]:
# ─── CELL 6: FIG 1 — MNIST VAE Loss ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(W_SINGLE, 2.2))
ax.plot(epochs_ax, mnist_recon, color=WARM, lw=1.2, label='Reconstruction')
ax.plot(epochs_ax, mnist_kl,    color=COOL, lw=1.2, linestyle='--', label='KL Divergence')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (per sample)')
ax.set_title('Fig. 1 - MNIST VAE Training Loss')
ax.legend(framealpha=0.9)
ax.grid(True, lw=0.3, alpha=0.5)
ax.set_xlim(1, 12)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_01.png', facecolor='white')
plt.show()
print('Saved fig_01.png')


Saved fig_01.png


In [7]:
# ─── CELL 7: FIG 2 — Fashion VAE Loss ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(W_SINGLE, 2.2))
ax.plot(epochs_ax, fashion_recon, color=WARM, lw=1.2, label='Reconstruction')
ax.plot(epochs_ax, fashion_kl,    color=COOL, lw=1.2, linestyle='--', label='KL Divergence')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (per sample)')
ax.set_title('Fig. 2 - Fashion-MNIST VAE Training Loss')
ax.legend(framealpha=0.9)
ax.grid(True, lw=0.3, alpha=0.5)
ax.set_xlim(1, 12)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_02.png', facecolor='white')
plt.show()
print('Saved fig_02.png')


Saved fig_02.png


In [8]:
# ─── CELL 8: FIG 3 — MNIST CNN Accuracy ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(W_SINGLE, 2.2))
ax.plot(cnn_epochs, cnn_mnist_acc, color=WARM, lw=1.2, marker='o', ms=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Fig. 3 - MNIST CNN Accuracy')
ax.set_ylim(94, 100)
ax.grid(True, lw=0.3, alpha=0.5)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_03.png', facecolor='white')
plt.show()
print('Saved fig_03.png')


Saved fig_03.png


In [9]:
# ─── CELL 9: FIG 4 — Fashion CNN Accuracy ────────────────────────────────────
fig, ax = plt.subplots(figsize=(W_SINGLE, 2.2))
ax.plot(cnn_epochs, cnn_fashion_acc, color=COOL, lw=1.2, marker='o', ms=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Fig. 4 - Fashion-MNIST CNN Accuracy')
ax.set_ylim(82, 96)
ax.grid(True, lw=0.3, alpha=0.5)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_04.png', facecolor='white')
plt.show()
print('Saved fig_04.png')


Saved fig_04.png


In [10]:
# ─── CELL 10: FIG 5 — MNIST Original vs Reconstructed ────────────────────────

def recon_grid(vae, loader, title, fname, n_cols=10):
    imgs, _ = next(iter(loader))
    imgs = imgs[:n_cols].to(device)
    with torch.no_grad():
        recon, _, _ = vae(imgs)       # 28x28 out
    fig, axes = plt.subplots(2, n_cols, figsize=(W_DOUBLE, 1.4))
    for i in range(n_cols):
        axes[0, i].imshow((imgs[i,0].cpu().numpy()  + 1) / 2, cmap='gray', vmin=0, vmax=1)
        axes[1, i].imshow((recon[i,0].cpu().numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1)
        for ax in axes[:, i]: ax.axis('off')
    axes[0, 0].set_ylabel('Orig',  fontsize=6)
    axes[1, 0].set_ylabel('Recon', fontsize=6)
    fig.suptitle(title, fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fig.savefig(fname, facecolor='white')
    plt.show()
    print(f'Saved {fname}')

recon_grid(vae_mnist, mnist_loader,
           title='Fig. 5 - MNIST: Original vs Reconstructed',
           fname=OUT_DIR / 'fig_05.png')


Saved ieee_outputs/fig_05.png


In [11]:
# ─── CELL 11: FIG 6 — Fashion Original vs Reconstructed ──────────────────────
recon_grid(vae_fashion, fashion_loader,
           title='Fig. 6 - Fashion-MNIST: Original vs Reconstructed',
           fname=OUT_DIR / 'fig_06.png')


Saved ieee_outputs/fig_06.png


In [12]:
# ─── CELL 12: FIG 7 — MNIST Latent Traversal ──────────────────────────────────

def latent_traversal(vae, latent_bank, title, fname, n_dims=6, n_steps=9):
    anchor = latent_bank[0].mean(0).to(device)
    values = np.linspace(-2, 2, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(W_DOUBLE, n_dims * 0.55))
    with torch.no_grad():
        for row, dim in enumerate(range(n_dims)):
            for col, v in enumerate(values):
                z = anchor.clone().unsqueeze(0)
                z[0, dim] = v
                img_np = (vae.decoder(z)[0, 0].cpu().numpy() + 1) / 2
                axes[row, col].imshow(img_np, cmap='gray', vmin=0, vmax=1)
                axes[row, col].axis('off')
            axes[row, 0].set_ylabel(f'z[{dim}]', fontsize=5, labelpad=2)
    for col, v in enumerate(values):
        axes[0, col].set_title(f'{v:.1f}', fontsize=5, pad=1)
    fig.suptitle(title, fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(fname, facecolor='white')
    plt.show()
    print(f'Saved {fname}')

latent_traversal(vae_mnist, lb_mnist,
                 title='Fig. 7 - MNIST Latent Traversal (6 dims)',
                 fname=OUT_DIR / 'fig_07.png')


Saved ieee_outputs/fig_07.png


In [13]:
# ─── CELL 13: FIG 8 — Fashion Latent Traversal ────────────────────────────────
latent_traversal(vae_fashion, lb_fashion,
                 title='Fig. 8 - Fashion-MNIST Latent Traversal (6 dims)',
                 fname=OUT_DIR / 'fig_08.png')


Saved ieee_outputs/fig_08.png


In [14]:
# ─── CELL 14: FIG 9 — MNIST Class-Anchored Generation ────────────────────────

def class_anchored_grid(vae, latent_bank, title, fname, n_per_class=8, class_names=None):
    fig, axes = plt.subplots(10, n_per_class, figsize=(W_DOUBLE, 10 * 0.55))
    with torch.no_grad():
        for cls in range(10):
            mu    = latent_bank[cls].mean(0).to(device)
            noise = torch.randn(n_per_class, vae.latent_dim).to(device) * 0.4
            z     = mu.unsqueeze(0) + noise
            imgs  = vae.decoder(z)
            for i in range(n_per_class):
                img_np = (imgs[i, 0].cpu().numpy() + 1) / 2
                axes[cls, i].imshow(img_np, cmap='gray', vmin=0, vmax=1)
                axes[cls, i].axis('off')
            lbl = class_names[cls] if class_names else str(cls)
            axes[cls, 0].set_ylabel(lbl, fontsize=5, labelpad=2)
    fig.suptitle(title, fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(fname, facecolor='white')
    plt.show()
    print(f'Saved {fname}')

class_anchored_grid(vae_mnist, lb_mnist,
                    title='Fig. 9 - MNIST Class-Anchored Generation',
                    fname=OUT_DIR / 'fig_09.png')


Saved ieee_outputs/fig_09.png


In [15]:
# ─── CELL 15: FIG 10 — Fashion Class-Anchored Generation ─────────────────────
class_anchored_grid(vae_fashion, lb_fashion,
                    title='Fig. 10 - Fashion-MNIST Class-Anchored Generation',
                    fname=OUT_DIR / 'fig_10.png',
                    class_names=FASHION_NAMES)


Saved ieee_outputs/fig_10.png


In [16]:
# ─── CELL 16: FIG 11 — Grad-CAM ──────────────────────────────────────────────
# VAE outputs 28x28. CNN (ResNet18) needs >= 32px so we upscale for CNN only.
# CAM is resized back to 28x28 to overlay on original generated image.

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.grads = None
        self.acts  = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'acts', o))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'grads', go[0]))

    def generate(self, x, class_idx):
        self.model.zero_grad()
        out = self.model(x)
        out[0, class_idx].backward()
        w   = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(28, 28), mode='bilinear', align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam


gcam         = GradCAM(cnn_fashion, cnn_fashion.layer4[-1])
SHOW_CLASSES = [0, 2, 6, 7, 8]
n_show       = len(SHOW_CLASSES)

fig, axes = plt.subplots(3, n_show, figsize=(W_DOUBLE, 2.6))

for col, cls in enumerate(SHOW_CLASSES):
    mu = lb_fashion[cls].mean(0).to(device)
    z  = (mu + torch.randn_like(mu) * 0.3).unsqueeze(0)

    with torch.no_grad():
        img_raw = vae_fashion.decoder(z)    # (1,1,28,28)

    # Upscale to 64x64 for CNN only
    img_cnn = F.interpolate(img_raw, size=(64, 64), mode='bilinear', align_corners=False)
    img_cnn = img_cnn.clone().requires_grad_(True)
    cam     = gcam.generate(img_cnn, cls)   # returns 28x28

    img_np = (img_raw[0, 0].detach().cpu().numpy() + 1) / 2

    axes[0, col].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(FASHION_NAMES[cls], fontsize=6)
    axes[0, col].axis('off')

    axes[1, col].imshow(cam, cmap='RdYlBu_r', vmin=0, vmax=1)
    axes[1, col].axis('off')

    overlay = np.stack([img_np, img_np, img_np], axis=-1)
    heatmap = plt.cm.RdYlBu_r(cam)[:, :, :3]
    blended = np.clip(0.55 * overlay + 0.45 * heatmap, 0, 1)
    axes[2, col].imshow(blended)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Generated', fontsize=5)
axes[1, 0].set_ylabel('Grad-CAM',  fontsize=5)
axes[2, 0].set_ylabel('Overlay',   fontsize=5)

fig.suptitle('Fig. 11 - Grad-CAM Attribution (Fashion-MNIST)', fontsize=8)
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUT_DIR / 'fig_11.png', facecolor='white')
plt.show()
print('Saved fig_11.png')


Saved fig_11.png


In [19]:
# ─── CELL 17: FIG 12 — t-SNE Latent Space ────────────────────────────────────

def get_latents(vae, loader, max_samples=2000):
    zs, ys = [], []
    total  = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)          # 28x28 direct
            mu, _ = vae.encode(x)
            zs.append(mu.cpu().numpy())
            ys.append(y.numpy())
            total += x.size(0)
            if total >= max_samples:
                break
    return np.concatenate(zs)[:max_samples], np.concatenate(ys)[:max_samples]

print('Collecting latent vectors...')
z_mnist,   y_mnist   = get_latents(vae_mnist,   mnist_loader)
z_fashion, y_fashion = get_latents(vae_fashion, fashion_loader)

print('Running t-SNE...')
emb_mnist = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42,
    max_iter=500
).fit_transform(z_mnist)

emb_fashion = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42,
    max_iter=500
).fit_transform(z_fashion)
cmap10 = plt.cm.get_cmap('RdYlBu', 10)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(W_DOUBLE, 2.8))

for cls in range(10):
    mask = y_mnist == cls
    ax1.scatter(emb_mnist[mask, 0], emb_mnist[mask, 1],
                s=1.5, color=cmap10(cls), label=str(cls), alpha=0.7)
ax1.set_title('MNIST', fontsize=8)
ax1.axis('off')
ax1.legend(title='Digit', fontsize=4, title_fontsize=5,
           markerscale=3, loc='lower right', ncol=2, framealpha=0.8)

for cls in range(10):
    mask = y_fashion == cls
    ax2.scatter(emb_fashion[mask, 0], emb_fashion[mask, 1],
                s=1.5, color=cmap10(cls), label=FASHION_NAMES[cls], alpha=0.7)
ax2.set_title('Fashion-MNIST', fontsize=8)
ax2.axis('off')
ax2.legend(title='Class', fontsize=4, title_fontsize=5,
           markerscale=3, loc='lower right', ncol=2, framealpha=0.8)

fig.suptitle('Fig. 12 - t-SNE Latent Space Visualization', fontsize=8)
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUT_DIR / 'fig_12.png', facecolor='white')
plt.show()
print('Saved fig_12.png')


Running t-SNE...
Saved fig_12.png


In [20]:
# ─── CELL 18: Summary ─────────────────────────────────────────────────────────
saved = sorted(OUT_DIR.glob('*.png'))
print(f'\n {len(saved)} figures saved to: {OUT_DIR.resolve()}\n')
for f in saved:
    sz = f.stat().st_size / 1024
    print(f'  {f.name}  ({sz:.0f} KB)')



 12 figures saved to: /workspace/notebooks/04_evaluation/ieee_outputs

  fig_01.png  (60 KB)
  fig_02.png  (62 KB)
  fig_03.png  (44 KB)
  fig_04.png  (48 KB)
  fig_05.png  (33 KB)
  fig_06.png  (44 KB)
  fig_07.png  (80 KB)
  fig_08.png  (90 KB)
  fig_09.png  (100 KB)
  fig_10.png  (126 KB)
  fig_11.png  (50 KB)
  fig_12.png  (451 KB)
